In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [3]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [4]:
# FIX (Bug 2): Use the Instruct model, not the base model.
# The base model ("Meta-Llama-3.1-8B") won't reliably follow instructions
# or produce structured output. The Instruct variant was fine-tuned for that.
#
# FIX (H10 - frontier-vs-legacy generation parity): upgraded the Meta-family
# generator from Llama 3.1 (mid-2024) to the current Llama 4 generation, so it is
# a comparable *generation* to the frontier GPT generator instead of confounding
# "model family" with "model generation".
#   Llama 4 is MoE-only - there is no small dense Llama-4 8B - so the smallest
#   current-gen Llama is Scout (17B active / 109B total, 16 experts).
#   WARNING - HARDWARE: the 4-bit Scout checkpoint is ~55 GB and will NOT fit a
#   free Colab T4 (16 GB). Run this on an A100 80GB / H100 (Colab Pro+ or equiv).
from unsloth import FastLanguageModel
import torch

# FIX (Bug C5): max_seq_length must hold BOTH the input prompt and the generated
# output. The old 2048 cap left only ~48 tokens for the article once
# max_new_tokens reserved output space, truncating the article to nothing.
# Llama 4 Scout has a very long native context, so we raise the cap to feed whole
# arXiv articles. KV-cache memory scales with the tokens actually used (~2-3k for
# one article + a 400-token summary), not this ceiling, so the high cap is cheap.
max_seq_length = 131072
dtype = None          # Auto-detect: float16 for T4/V100, bfloat16 for Ampere+
load_in_4bit = True   # 4-bit quantization to fit in GPU memory

model, tokenizer = FastLanguageModel.from_pretrained(
    # Current-generation Meta model (Llama 4 Scout, Apr 2025), 4-bit via Unsloth.
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.7.5: Fast Llama patching. Transformers: 5.13.1.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


In [5]:
# FIX (Bug 2 continued): LoRA / PEFT setup REMOVED.
# get_peft_model() adds trainable adapter weights — that's for fine-tuning,
# not inference. We're using the Instruct model as-is, so we skip this entirely.

# Instead, put the model into fast inference mode:
FastLanguageModel.for_inference(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm):

In [7]:
df=pd.read_csv("ST-Bench_all_domain.csv")
print(f"Total rows in CSV: {len(df)}")

Total rows in CSV: 150


In [8]:
df.head()

,title,abstract,license,url,html_url,date
0,MediX-R1: Open Ended Medical Reinforcement Lea...,"We introduce MediX-R1, an open-ended Reinforce...",http://creativecommons.org/licenses/by-nc-sa/4.0/,https://arxiv.org/abs/2602.23363,https://arxiv.org/html/2602.23363,2026-02-26
1,Model Agreement via Anchoring,Numerous lines of aim to control $\textit{mode...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23360,https://arxiv.org/html/2602.23360,2026-02-26
2,A Dataset is Worth 1 MB,A dataset server must often distribute the sam...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23358,https://arxiv.org/html/2602.23358,2026-02-26
3,SOTAlign: Semi-Supervised Alignment of Unimoda...,The Platonic Representation Hypothesis posits ...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23353,https://arxiv.org/html/2602.23353,2026-02-26
4,Scale Can't Overcome Pragmatics: The Impact of...,The lack of reasoning capabilities in Vision-L...,http://creativecommons.org/licenses/by/4.0/,https://arxiv.org/abs/2602.23351,https://arxiv.org/html/2602.23351,2026-02-26


In [9]:
# --- pipeline bootstrap: single source of truth is pipeline.py ---
# Makes pipeline.py importable whether running from a local repo checkout or in
# Google Colab, then imports the shared article fetch/clean helpers. Edit the
# fetch/clean logic ONCE in pipeline.py - not here, and not per-notebook.
import os, sys

def _ensure_pipeline_importable():
    try:
        import pipeline  # noqa: F401
        return
    except ImportError:
        pass
    # Local checkout: walk up from the CWD looking for pipeline.py
    here = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(here, "pipeline.py")):
            sys.path.insert(0, here)
            return
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    # Colab / fresh runtime: fetch pipeline.py from the repo's main branch
    import urllib.request
    url = "https://raw.githubusercontent.com/Dorothy99-love/Style-transfer/main/pipeline.py"
    urllib.request.urlretrieve(url, "pipeline.py")
    sys.path.insert(0, os.getcwd())

_ensure_pipeline_importable()
from pipeline import fetch_html_body_content, get_article_snippet_without_abstract


In [10]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

In [11]:
import traceback
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
# from pipeline import fetch_html_body_content

# Initialize columns for the results
df["llama_response"] = ""  # Ensure the target column exists

In [12]:
!nvidia-smi

Fri Jul 24 08:02:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             55W /  400W |    6048MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# --- Main evaluation loop ---
from transfer_prompt import transfer_prompt
for index, row in df.head(3).iterrows():
    outputs = None
    try:
        html_url = row.get("html_url", None)
        if not html_url:
            continue

        # Fetch the content (assuming fetch_html_body_content is defined elsewhere)
        article, status = fetch_html_body_content(html_url)
        if status != "html_success":
            print(f"Row {index}: HTML fetch failed ({status})")
            continue

        article_snippet = get_article_snippet_without_abstract(article)
        #delete the blank spaces
        article_snippet = re.sub(r'\s*\n\s*', ' ', article_snippet).strip()
        prompt=transfer_prompt+"\n"+"Paper Content\n"+article_snippet
        # Build chat message structure
        messages = [
            {"role": "user", "content": prompt}
        ]

        encoded = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            add_generation_prompt=True,
            truncation=True,
            return_dict=True
        )

        # move to device safely
        encoded = {k: v.to(device) for k, v in encoded.items()}

        outputs = model.generate(
            input_ids=encoded["input_ids"],
            attention_mask=encoded.get("attention_mask"),
            max_new_tokens=1000,
            temperature=0.0,
            do_sample=False
        )

        # ✅ 正确截取生成部分
        input_length = encoded["input_ids"].shape[-1]
        generated_tokens = outputs[:, input_length:]

        output_text = tokenizer.decode(
            generated_tokens[0],
            skip_special_tokens=True
        ).strip()

        # Update the specific row's 'llama_response' column with the transferred text
        # Using .at[index, col] ensures we only modify the current row
        df.at[index, "llama_response"] = output_text

        print(f"--- Row {index}: Successfully processed ---")
        print("Input prompt:\n", prompt)
        print("Output:\n", output_text)

    except torch.cuda.OutOfMemoryError as e:
        print(f"Error processing Row {index}: {e}")

    finally:
        if outputs is not None:
            del outputs
        torch.cuda.empty_cache()

# Save the final results to CSV
df.to_csv("results_llama_all.csv", index=False)
print(f"\nSaved {len(df)} rows to results_llama_all.csv")

Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Row 0: Successfully processed ---
Input prompt:
 
You are a professional science popularization writer.

Task:
Rewrite the following scientific paper content into a clear and accessible popular-science article explanation, targeting the high school students. 

Requirements:

You should avoid equations and heavy jargons and the total length should be 150-200 words. Please consider the following three dimensions when responding. 
1.Style transfer intensity: the scientific content should been effectively transformed into a more accessible, popular-science style which referring to well-known popular science magazines such as "WIRED" or "National Geographic".
2.Content preservation: the summary should reflect the original article, including all major claims, methodologies, findings and contributions.
3.Language Naturalness: the summary should read like human-written text.


Paper Content
Introduction
Large medical language and vision-language models are increasingly deployed for clinica

Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Row 1: Successfully processed ---
Input prompt:
 
You are a professional science popularization writer.

Task:
Rewrite the following scientific paper content into a clear and accessible popular-science article explanation, targeting the high school students. 

Requirements:

You should avoid equations and heavy jargons and the total length should be 150-200 words. Please consider the following three dimensions when responding. 
1.Style transfer intensity: the scientific content should been effectively transformed into a more accessible, popular-science style which referring to well-known popular science magazines such as "WIRED" or "National Geographic".
2.Content preservation: the summary should reflect the original article, including all major claims, methodologies, findings and contributions.
3.Language Naturalness: the summary should read like human-written text.


Paper Content
Introduction
Two predictive models
f
,
f
:
𝒳
→
ℝ
f_{1},f_{2}:\mathcal{X}\rightarrow\mathbb{R}
, trai

Both `max_new_tokens` (=1000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Row 2: Successfully processed ---
Input prompt:
 
You are a professional science popularization writer.

Task:
Rewrite the following scientific paper content into a clear and accessible popular-science article explanation, targeting the high school students. 

Requirements:

You should avoid equations and heavy jargons and the total length should be 150-200 words. Please consider the following three dimensions when responding. 
1.Style transfer intensity: the scientific content should been effectively transformed into a more accessible, popular-science style which referring to well-known popular science magazines such as "WIRED" or "National Geographic".
2.Content preservation: the summary should reflect the original article, including all major claims, methodologies, findings and contributions.
3.Language Naturalness: the summary should read like human-written text.


Paper Content
Introduction
Sending training datasets from a central server to multiple clients is an expensive pro